In [7]:
import pymysql

conn = pymysql.connect(
    host='localhost',
    user='root',
    password='123456',
    database='comp7640_ecommerce',
    charset='utf8mb4'
)
# 建立游标，全程保持打开，直到最后关闭
cursor = conn.cursor()
print("✅ 数据库连接成功，游标已激活")

✅ 数据库连接成功，游标已激活


In [8]:
# 功能1：浏览每个商家的所有产品
sql = """
SELECT 
    v.vendor_id, v.business_name,
    p.product_id, p.name, p.listed_price, p.stock_quantity, p.tag1, p.tag2, p.tag3
FROM vendors v
LEFT JOIN products p ON v.vendor_id = p.vendor_id
ORDER BY v.vendor_id, p.product_id;
"""

# 执行查询
cursor.execute(sql)
results = cursor.fetchall()

# 格式化输出
print("="*90)
print("📋 各商家产品目录 (Browse all products by vendor)")
print("="*90)

current_vendor = None
for row in results:
    vid, vname, pid, pname, price, stock, t1, t2, t3 = row
    if current_vendor != vid:
        current_vendor = vid
        print(f"\n🏪 商家ID: {vid} | 名称: {vname}")
        print("-"*80)
    if pid:
        print(f"产品ID: {pid:3d} | {pname:<20} | 价格: {price:>8.2f} | 库存: {stock:>4d} | 标签: {t1}/{t2}/{t3}")
    else:
        print("⚠️ 该商家暂无产品")

📋 各商家产品目录 (Browse all products by vendor)

🏪 商家ID: 1 | 名称: Apple
--------------------------------------------------------------------------------
产品ID:   1 | IPhone 17 Pro        | 价格:  9999.00 | 库存: 9000 | 标签: electronics/phone/wireless
产品ID:   2 | IWatch Ultra         | 价格:  6799.00 | 库存: 8000 | 标签: electronics/wearable/fitness

🏪 商家ID: 2 | 名称: BYD
--------------------------------------------------------------------------------
产品ID:   3 | QING                 | 价格: 79999.00 | 库存: 10000 | 标签: electronics/car/domestic
产品ID:   4 | HAN                  | 价格: 200000.00 | 库存: 10000 | 标签: electronics/car/domestic

🏪 商家ID: 3 | 名称: SONY
--------------------------------------------------------------------------------
产品ID:   5 | alpha 3              | 价格: 50000.00 | 库存: 10000 | 标签: electronics/camera/kitchen
产品ID:   6 | Stainless Steel Kettle | 价格:   199.00 | 库存:   60 | 标签: electronics/camera/appliance


In [9]:
# 功能2：给指定商家添加新产品
# -------------------------- 可修改参数 --------------------------
vendor_id = 1          # 目标商家ID（对应vendors表的vendor_id）
product_name = "iPad Pro 2026"
price = 12999.00
stock = 500
tag1 = "electronics"
tag2 = "tablet"
tag3 = "wireless"
# ----------------------------------------------------------------

# 插入新产品SQL
sql_insert = """
INSERT INTO products (vendor_id, name, listed_price, stock_quantity, tag1, tag2, tag3)
VALUES (%s, %s, %s, %s, %s, %s, %s);
"""

# 执行插入，提交事务
cursor.execute(sql_insert, (vendor_id, product_name, price, stock, tag1, tag2, tag3))
conn.commit()
print(f"✅ 成功给商家{vendor_id}添加新产品：{product_name}")

# 插入后重新查询验证
print("\n🔍 插入后重新查询商家产品：")
cursor.execute(sql)
results = cursor.fetchall()
current_vendor = None
for row in results:
    vid, vname, pid, pname, price, stock, t1, t2, t3 = row
    if vid == vendor_id:
        if current_vendor != vid:
            current_vendor = vid
            print(f"\n🏪 商家ID: {vid} | 名称: {vname}")
            print("-"*80)
        print(f"产品ID: {pid:3d} | {pname:<20} | 价格: {price:>8.2f} | 库存: {stock:>4d} | 标签: {t1}/{t2}/{t3}")

✅ 成功给商家1添加新产品：iPad Pro 2026

🔍 插入后重新查询商家产品：

🏪 商家ID: 1 | 名称: Apple
--------------------------------------------------------------------------------
产品ID:   1 | IPhone 17 Pro        | 价格:  9999.00 | 库存: 9000 | 标签: electronics/phone/wireless
产品ID:   2 | IWatch Ultra         | 价格:  6799.00 | 库存: 8000 | 标签: electronics/wearable/fitness
产品ID:   7 | iPad Pro 2026        | 价格: 12999.00 | 库存:  500 | 标签: electronics/tablet/wireless


In [ ]:
# 所有功能执行完成后，再关闭游标和连接
cursor.close()
conn.close()
print("🔌 数据库连接已关闭，操作完成")